# 03 · Transformación y dataset final

**Transformación**

Se parte del archivo limpio de la Parte II (1.006 ofertas, 12 columnas) 
para dejar las variables en la forma que necesita el análisis: separar 
campos compuestos, tipar fechas correctamente, eliminar columnas sin 
valor analítico y documentar el dataset resultante en un diccionario 
de datos final. El proyecto usa una única fuente, por lo que esta etapa 
no integra datos externos.

**Resultado de esta transformación:** se separó `ubicacion` en 
`departamento` y `distrito`, y se convirtieron las fechas de publicación 
a tipo `datetime`. Se eliminaron cuatro columnas sin valor analítico para 
el objetivo del proyecto (`fuente_url`, `fecha_extraccion_utc`, 
`pagina_origen` y la propia `ubicacion`, ya redundante tras separarla). 
El dataset transformado conserva las 1.006 ofertas y queda con 10 
columnas. Se generó `docs/diccionario_datos_final.csv` con una fila por 
variable, siguiendo el mismo formato del diccionario inicial.

## Preparación

In [1]:
import sys
from pathlib import Path

# Ubica la raíz del repositorio (la carpeta que contiene src/config.py),
# así el notebook funciona desde notebooks/, desde la raíz o en Colab.
RAIZ = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists()), None)
if RAIZ is None:
    raise FileNotFoundError(
        "No se encontró la raíz del repositorio. Abra el notebook desde la carpeta del proyecto; "
        "en Colab, ejecute antes: !git clone <url del repositorio> y %cd <carpeta del repositorio>"
    )
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd
from src import config, validacion
from src.registro import Registro

registro = Registro("03_transformacion")
df = validacion.cargar_entrada("limpio")
filas_iniciales = len(df)

[OK] limpio: data\interim\talento_peru_limpio.csv (1006 filas, 12 columnas)


## 1. Integración de fuentes

No aplica. El proyecto usa una única fuente (Talento Perú - SERVIR), según se documenta en el README. Esta etapa se enfoca en transformación, reducción y generación del dataset final a partir de `talento_peru_limpio.csv`.

## 2. Transformación de variables

### Separación de ubicación

La columna `ubicacion` llega en formato `"DEPARTAMENTO - DISTRITO"` 
(observado en el notebook de extracción). La separamos en dos columnas 
nuevas, `departamento` y `distrito`, para poder analizar y agrupar por 
cada nivel geográfico de forma independiente.

In [2]:
partes = df["ubicacion"].str.split(" - ", n=1, expand=True)
df["departamento"] = partes[0].str.strip()
df["distrito"] = partes[1].str.strip()

sin_separador = df["distrito"].isna().sum()
print("Filas sin distrito separable:", sin_separador)

registro.agregar(
    "Separación de ubicación",
    "Se dividió 'ubicacion' en 'departamento' y 'distrito' usando ' - ' como separador",
    int(sin_separador),
    "El formato 'DEPARTAMENTO - DISTRITO' se observó consistente en el dataset limpio."
)

df[["ubicacion", "departamento", "distrito"]].head(5)

Filas sin distrito separable: 0


,ubicacion,departamento,distrito
0,JUNIN - EL TAMBO,JUNIN,EL TAMBO
1,JUNIN - EL TAMBO,JUNIN,EL TAMBO
2,JUNIN - EL TAMBO,JUNIN,EL TAMBO
3,CUSCO - SICUANI,CUSCO,SICUANI
4,CUSCO - SICUANI,CUSCO,SICUANI


### Conversión de fechas a tipo datetime

`fecha_inicio_publicacion` y `fecha_fin_publicacion` llegan como texto 
(`"AAAA-MM-DD"`) desde la etapa de limpieza. Las convertimos a tipo 
`datetime` para poder calcular duraciones y operar con ellas en las 
siguientes etapas (por ejemplo, análisis de vigencia de las convocatorias).

In [3]:
for col in config.COLUMNAS_FECHA:
    df[col] = pd.to_datetime(df[col], format="%Y-%m-%d")

registro.agregar(
    "Conversión de fechas",
    "Se convirtieron fecha_inicio_publicacion y fecha_fin_publicacion de texto a datetime",
    0,
    "Las fechas ya venían validadas y consistentes desde la etapa de limpieza (formato AAAA-MM-DD)."
)

print(df[config.COLUMNAS_FECHA].dtypes)
df[config.COLUMNAS_FECHA].head(3)

fecha_inicio_publicacion    datetime64[ns]
fecha_fin_publicacion       datetime64[ns]
dtype: object


,fecha_inicio_publicacion,fecha_fin_publicacion
0,2026-09-18,2026-10-01
1,2026-09-18,2026-10-01
2,2026-09-18,2026-10-01


## 3. Reducción de datos

### Eliminación de columnas sin valor analítico

Revisamos las columnas puramente técnicas/de trazabilidad (`fuente_url`, 
`fecha_extraccion_utc`, `pagina_origen`) para decidir si aportan al 
análisis final o si conviene dejarlas fuera del dataset transformado.

In [4]:
for col in ["fuente_url", "fecha_extraccion_utc", "pagina_origen"]:
    print(col, "->", df[col].nunique(), "valores distintos")
    print(df[col].unique()[:3], "\n")

fuente_url -> 1 valores distintos
['https://app.servir.gob.pe/DifusionOfertasExterno/faces/consultas/ofertas_laborales.xhtml'] 

fecha_extraccion_utc -> 105 valores distintos
['2026-09-18T05:58:55.931516+00:00' '2026-09-18T05:58:59.842714+00:00'
 '2026-09-18T05:59:03.730899+00:00'] 

pagina_origen -> 105 valores distintos
[1 2 3] 



Se decidió proceder con la eliminación de esas tres columnas, dado que no aportan al análisis del mercado laboral, aunque fueron útiles durante la extracción.

- **`fuente_url`**: tiene un único valor en las 1006 filas (la misma URL 
  del portal para todas las ofertas). No distingue ni aporta información 
  entre registros.
- **`fecha_extraccion_utc`**: marca el momento exacto en que el scraper 
  leyó cada página del portal, no una característica de la oferta laboral. 
  Es metadata del proceso de extracción (Parte I del proyecto), no del fenómeno   que se analiza.
- **`pagina_origen`**: indica en qué página del listado paginado apareció 
  la oferta. Es un artefacto de cómo se recorrió el sitio web, sin relación 
  con el perfil o las características del puesto.

Cabe mencionar que las columnas no se pierden del proyecto, siguen disponibles en `data/raw/talento_peru_ofertas.csv` y en `data/interim/talento_peru_limpio.csv` por si alguna etapa posterior las necesitara. Aquí solo se excluyen de la salida de esta etapa (`talento_peru_transformado.csv`) porque no son relevantes para el análisis final.

In [5]:
columnas_a_eliminar = ["fuente_url", "fecha_extraccion_utc", "pagina_origen"]
columnas_antes = df.shape[1]

df = df.drop(columns=columnas_a_eliminar)

registro.agregar(
    "Reducción de columnas",
    f"Se eliminaron columnas sin valor analítico: {', '.join(columnas_a_eliminar)}",
    columnas_antes - df.shape[1],
    "fuente_url tiene un único valor en todo el dataset; fecha_extraccion_utc y "
    "pagina_origen son metadata del proceso de scraping, no del fenómeno analizado. "
    "Las tres columnas permanecen disponibles en etapas anteriores del pipeline."
)

print(f"Columnas: {columnas_antes} → {df.shape[1]}")
df.columns.tolist()

Columnas: 14 → 11


['id_oferta_hash',
 'puesto',
 'entidad',
 'ubicacion',
 'numero_convocatoria',
 'vacantes',
 'remuneracion_pen',
 'fecha_inicio_publicacion',
 'fecha_fin_publicacion',
 'departamento',
 'distrito']

### Eliminación de `ubicacion`

Finalmente, tras separar `ubicacion` en `departamento` y `distrito`, la columna original queda redundante; es decir, contiene exactamente la misma información, solo que sin dividir. Por ello, se elimina para evitar duplicidad en el dataset final.

In [6]:
columnas_antes = df.shape[1]

df = df.drop(columns=["ubicacion"])

registro.agregar(
    "Eliminación de ubicacion",
    "Se eliminó la columna original 'ubicacion' tras separarla en 'departamento' y 'distrito'",
    columnas_antes - df.shape[1],
    "La información queda duplicada una vez separada; conservar ambas versiones no aporta valor adicional."
)

print(f"Columnas: {columnas_antes} → {df.shape[1]}")
df.columns.tolist()

Columnas: 11 → 10


['id_oferta_hash',
 'puesto',
 'entidad',
 'numero_convocatoria',
 'vacantes',
 'remuneracion_pen',
 'fecha_inicio_publicacion',
 'fecha_fin_publicacion',
 'departamento',
 'distrito']

### Diccionario de datos final

Se documenta cada variable del dataset transformado en 
`docs/diccionario_datos_final.csv`, siguiendo el mismo formato que 
`docs/diccionario_datos_inicial.csv`.

### Revisión del diccionario final existente

Se revisa la estructura de `docs/diccionario_datos_final.csv` antes de 
completarlo.

In [7]:
pd.read_csv(config.DICCIONARIO_FINAL)

,variable,descripcion,tipo,tipo_pandas,tamano_maximo,valores,unidad
0,id_oferta_hash,"Identificador único de la oferta, generado con...",Categórico nominal,string,24,Código hexadecimal (1006 valores únicos); ej.:...,-
1,puesto,Nombre del puesto ofertado,Categórico nominal,string,127,838 valores distintos; ej.: ESPECIALISTA LEGAL...,-
2,entidad,Entidad pública que publica la oferta,Categórico nominal,string,100,113 valores distintos; ej.: CONTRALORIA GENERA...,-
3,numero_convocatoria,Código de la convocatoria asignado por la entidad,Categórico nominal,string,53,"282 valores distintos; ej.: 728-06, 276-003",-
4,vacantes,Cantidad de puestos disponibles en la oferta,Numérico de proporción,int64,2,1 a 72,Puestos
5,remuneracion_pen,Remuneración mensual ofrecida,Numérico de proporción,float64,8,1126.70 a 21000.00,Soles (S/) por mes
6,fecha_inicio_publicacion,Fecha en que la oferta empieza a difundirse,Numérico de intervalo,datetime64[ns],10,2026-09-17 a 2026-09-18,Fecha (AAAA-MM-DD)
7,fecha_fin_publicacion,Fecha en que termina la difusión de la oferta,Numérico de intervalo,datetime64[ns],10,2026-09-30 a 2026-10-01,Fecha (AAAA-MM-DD)
8,departamento,Departamento del Perú donde se ubica el puesto...,Categórico nominal,string,13,"23 valores distintos; ej.: LIMA, CUSCO, JUNIN",-
9,distrito,"Distrito donde se ubica el puesto, obtenido al...",Categórico nominal,string,22,"131 valores distintos; ej.: EL TAMBO, SICUANI",-


### Recalcular estadísticas sobre el dataset transformado

Las estadísticas del diccionario inicial (valores distintos, rangos) se 
calcularon sobre las 1010 filas originales. Como la limpieza retiró 4 
ofertas y esta etapa separó `ubicacion`, es necesario recalcular esos 
valores sobre el dataset actual (1006 filas) y obtener las estadísticas 
de las variables nuevas (`departamento`, `distrito`) antes de documentarlas.

In [8]:
print("departamento:", df["departamento"].nunique(), "valores distintos")
print(sorted(df["departamento"].unique())[:5], "...")
print()
print("distrito:", df["distrito"].nunique(), "valores distintos")
print(sorted(df["distrito"].unique())[:5], "...")
print()
print("puesto:", df["puesto"].nunique(), "valores distintos (dataset: 1006 filas)")
print("entidad:", df["entidad"].nunique(), "valores distintos")
print("numero_convocatoria:", df["numero_convocatoria"].nunique(), "valores distintos")
print("vacantes:", df["vacantes"].min(), "a", df["vacantes"].max())
print("remuneracion_pen:", df["remuneracion_pen"].min(), "a", df["remuneracion_pen"].max())
print("fechas inicio:", df["fecha_inicio_publicacion"].min(), "a", df["fecha_inicio_publicacion"].max())
print("fechas fin:", df["fecha_fin_publicacion"].min(), "a", df["fecha_fin_publicacion"].max())

departamento: 23 valores distintos
['AMAZONAS', 'ANCASH', 'APURIMAC', 'AREQUIPA', 'AYACUCHO'] ...

distrito: 131 valores distintos
['ABANCAY', 'AGALLPAMPA', 'AJOYANI', 'ALTO PICHIGUA', 'AMARILIS'] ...

puesto: 838 valores distintos (dataset: 1006 filas)
entidad: 113 valores distintos
numero_convocatoria: 282 valores distintos
vacantes: 1 a 72
remuneracion_pen: 1126.7 a 21000.0
fechas inicio: 2026-09-17 00:00:00 a 2026-09-18 00:00:00
fechas fin: 2026-09-30 00:00:00 a 2026-10-01 00:00:00


Se calcula la longitud máxima de cada variable categórica (`tamano_maximo` 
en el diccionario) sobre el dataset transformado, ya que estos valores 
también pudieron cambiar al eliminarse filas y crearse columnas nuevas.

In [9]:
columnas_texto = ["id_oferta_hash", "puesto", "entidad", "numero_convocatoria", "departamento", "distrito"]
for col in columnas_texto:
    print(col, "-> longitud máxima:", df[col].str.len().max())

id_oferta_hash -> longitud máxima: 24
puesto -> longitud máxima: 127
entidad -> longitud máxima: 100
numero_convocatoria -> longitud máxima: 53
departamento -> longitud máxima: 13
distrito -> longitud máxima: 22


### Construcción del diccionario de datos final

Se arma `docs/diccionario_datos_final.csv` con las 10 variables del 
dataset transformado, usando el mismo formato del diccionario inicial. 
Las estadísticas (valores únicos, rangos, tamaños máximos) corresponden 
a las 1006 filas resultantes de la limpieza, e incluyen `departamento` 
y `distrito` como variables nuevas de esta etapa.

In [10]:
diccionario_final = pd.DataFrame([
    {
        "variable": "id_oferta_hash",
        "descripcion": "Identificador único de la oferta, generado con SHA-256 a partir de sus 8 campos principales y recortado a 24 caracteres",
        "tipo": "Categórico nominal", "tipo_pandas": "string", "tamano_maximo": 24,
        "valores": "Código hexadecimal (1006 valores únicos); ej.: 1af30dbb2f0f39e7d0c908cf", "unidad": "-"
    },
    {
        "variable": "puesto",
        "descripcion": "Nombre del puesto ofertado",
        "tipo": "Categórico nominal", "tipo_pandas": "string", "tamano_maximo": 127,
        "valores": "838 valores distintos; ej.: ESPECIALISTA LEGAL, ASISTENTE ADMINISTRATIVO", "unidad": "-"
    },
    {
        "variable": "entidad",
        "descripcion": "Entidad pública que publica la oferta",
        "tipo": "Categórico nominal", "tipo_pandas": "string", "tamano_maximo": 100,
        "valores": "113 valores distintos; ej.: CONTRALORIA GENERAL DE LA REPUBLICA, PODER JUDICIAL", "unidad": "-"
    },
    {
        "variable": "numero_convocatoria",
        "descripcion": "Código de la convocatoria asignado por la entidad",
        "tipo": "Categórico nominal", "tipo_pandas": "string", "tamano_maximo": 53,
        "valores": "282 valores distintos; ej.: 728-06, 276-003", "unidad": "-"
    },
    {
        "variable": "vacantes",
        "descripcion": "Cantidad de puestos disponibles en la oferta",
        "tipo": "Numérico de proporción", "tipo_pandas": "int64", "tamano_maximo": 2,
        "valores": "1 a 72", "unidad": "Puestos"
    },
    {
        "variable": "remuneracion_pen",
        "descripcion": "Remuneración mensual ofrecida",
        "tipo": "Numérico de proporción", "tipo_pandas": "float64", "tamano_maximo": 8,
        "valores": "1126.70 a 21000.00", "unidad": "Soles (S/) por mes"
    },
    {
        "variable": "fecha_inicio_publicacion",
        "descripcion": "Fecha en que la oferta empieza a difundirse",
        "tipo": "Numérico de intervalo", "tipo_pandas": "datetime64[ns]", "tamano_maximo": 10,
        "valores": "2026-09-17 a 2026-09-18", "unidad": "Fecha (AAAA-MM-DD)"
    },
    {
        "variable": "fecha_fin_publicacion",
        "descripcion": "Fecha en que termina la difusión de la oferta",
        "tipo": "Numérico de intervalo", "tipo_pandas": "datetime64[ns]", "tamano_maximo": 10,
        "valores": "2026-09-30 a 2026-10-01", "unidad": "Fecha (AAAA-MM-DD)"
    },
    {
        "variable": "departamento",
        "descripcion": "Departamento del Perú donde se ubica el puesto, obtenido al separar 'ubicacion'",
        "tipo": "Categórico nominal", "tipo_pandas": "string", "tamano_maximo": 13,
        "valores": "23 valores distintos; ej.: LIMA, CUSCO, JUNIN", "unidad": "-"
    },
    {
        "variable": "distrito",
        "descripcion": "Distrito donde se ubica el puesto, obtenido al separar 'ubicacion'",
        "tipo": "Categórico nominal", "tipo_pandas": "string", "tamano_maximo": 22,
        "valores": "131 valores distintos; ej.: EL TAMBO, SICUANI", "unidad": "-"
    },
])

diccionario_final.to_csv(config.DICCIONARIO_FINAL, index=False, encoding=config.ENCODING)

registro.agregar(
    "Diccionario de datos final",
    "Se generó docs/diccionario_datos_final.csv con las 10 variables del dataset transformado",
    len(diccionario_final),
    "Se recalculan estadísticas sobre las 1006 filas resultantes de la limpieza y se documentan "
    "las variables nuevas (departamento, distrito) creadas en esta etapa."
)

diccionario_final

,variable,descripcion,tipo,tipo_pandas,tamano_maximo,valores,unidad
0,id_oferta_hash,"Identificador único de la oferta, generado con...",Categórico nominal,string,24,Código hexadecimal (1006 valores únicos); ej.:...,-
1,puesto,Nombre del puesto ofertado,Categórico nominal,string,127,838 valores distintos; ej.: ESPECIALISTA LEGAL...,-
2,entidad,Entidad pública que publica la oferta,Categórico nominal,string,100,113 valores distintos; ej.: CONTRALORIA GENERA...,-
3,numero_convocatoria,Código de la convocatoria asignado por la entidad,Categórico nominal,string,53,"282 valores distintos; ej.: 728-06, 276-003",-
4,vacantes,Cantidad de puestos disponibles en la oferta,Numérico de proporción,int64,2,1 a 72,Puestos
5,remuneracion_pen,Remuneración mensual ofrecida,Numérico de proporción,float64,8,1126.70 a 21000.00,Soles (S/) por mes
6,fecha_inicio_publicacion,Fecha en que la oferta empieza a difundirse,Numérico de intervalo,datetime64[ns],10,2026-09-17 a 2026-09-18,Fecha (AAAA-MM-DD)
7,fecha_fin_publicacion,Fecha en que termina la difusión de la oferta,Numérico de intervalo,datetime64[ns],10,2026-09-30 a 2026-10-01,Fecha (AAAA-MM-DD)
8,departamento,Departamento del Perú donde se ubica el puesto...,Categórico nominal,string,13,"23 valores distintos; ej.: LIMA, CUSCO, JUNIN",-
9,distrito,"Distrito donde se ubica el puesto, obtenido al...",Categórico nominal,string,22,"131 valores distintos; ej.: EL TAMBO, SICUANI",-


## 5. Registro de decisiones

In [11]:
registro.tabla()

,etapa,n,paso,accion,filas_afectadas,justificacion,fecha
0,03_transformacion,1,Separación de ubicación,Se dividió 'ubicacion' en 'departamento' y 'di...,0,El formato 'DEPARTAMENTO - DISTRITO' se observ...,2026-11-12
1,03_transformacion,2,Conversión de fechas,Se convirtieron fecha_inicio_publicacion y fec...,0,Las fechas ya venían validadas y consistentes ...,2026-11-12
2,03_transformacion,3,Reducción de columnas,Se eliminaron columnas sin valor analítico: fu...,3,fuente_url tiene un único valor en todo el dat...,2026-11-12
3,03_transformacion,4,Eliminación de ubicacion,Se eliminó la columna original 'ubicacion' tra...,1,La información queda duplicada una vez separad...,2026-11-12
4,03_transformacion,5,Diccionario de datos final,Se generó docs/diccionario_datos_final.csv con...,10,Se recalculan estadísticas sobre las 1006 fila...,2026-11-12


## 6. Guardado y validación

In [12]:
print(f"Filas: {filas_iniciales} → {len(df)} | Columnas: {df.shape[1]}")
config.asegurar_carpetas()
df.to_csv(config.ARCHIVO_TRANSFORMADO, index=False, encoding=config.ENCODING)
registro.guardar()
validacion.validar("transformado").exigir();

Filas: 1006 → 1006 | Columnas: 10
[OK] transformado: data\interim\talento_peru_transformado.csv (1006 filas, 10 columnas)
